# Exploring Extra Raw Data Sources

This notebook surveys the new `.raw` file collections in `extra_raw/` and `Datasets/extra_raw_files/`, comparing their properties (channels, frequencies, sample counts, datagram structure) against the existing training data. Each file is also rendered as a PNG echogram, organised by dataset and frequency.

In [ ]:
import sys, os, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict

sys.path.insert(0, '../..')
from project_paths import DATA_ROOT
from raw_sensor_integration import raw_parser, ping_parser

warnings.filterwarnings('ignore')

BASE = DATA_ROOT
OUT_DIR = BASE / 'Master' / 'Datasets' / 'extra_raw_echograms'
OUT_DIR.mkdir(exist_ok=True)

## Dataset definitions

The existing training data uses a dual-frequency combo transducer (`ES38-18|200-18CR`) with two channels at 38 kHz and 200 kHz. The new datasets come from various vessels and surveys with different transducer configurations.

In [ ]:
DATASETS = {
    'first_chrome': BASE / 'extra_raw' / 'first_chrome',
    'EK80_5': BASE / 'extra_raw' / 'TransferNow-RawFilesEK80_5',
    'EK80_6': BASE / 'extra_raw' / 'TransferNow-RawFilesEK80_6',
    'split_single': BASE / 'Master' / 'Datasets' / 'extra_raw_files' / '2025-02-05 Split as Single - original files pcap sim not averaged',
    'basto1': BASE / 'Master' / 'Datasets' / 'extra_raw_files' / 'basto1',
    'diffraw': BASE / 'Master' / 'Datasets' / 'extra_raw_files' / 'diffraw',
    'EK80_3_noise': BASE / 'Master' / 'Datasets' / 'extra_raw_files' / 'TransferNow-RawFilesEK80_3',
    'EK80_3_testrec': BASE / 'Master' / 'Datasets' / 'extra_raw_files' / 'TransferNow-RawFilesEK80_3' / 'TestRecRaw',
    'EK80_3_impedance': BASE / 'Master' / 'Datasets' / 'extra_raw_files' / 'TransferNow-RawFilesEK80_3' / 'ImpedanceRaw',
    'EK80_3_calibration': BASE / 'Master' / 'Datasets' / 'extra_raw_files' / 'TransferNow-RawFilesEK80_3' / 'C1C2CalibrationRaw',
    'EK80_4': BASE / 'Master' / 'Datasets' / 'extra_raw_files' / 'TransferNow-RawFilesEK80_4',
}

EXISTING_REF = BASE / 'Master' / 'Datasets' / 'raw_files' / 'drive-download-20260317T122216Z-1-001'

def list_raw_files(directory):
    return sorted([
        directory / f for f in os.listdir(directory)
        if f.endswith('.raw') and not f.startswith('._')
    ])

for name, path in DATASETS.items():
    try:
        files = list_raw_files(path)
        total_mb = sum(f.stat().st_size for f in files) / 1e6
        print(f'{name:20s}  {len(files):3d} files  {total_mb:8.1f} MB')
    except FileNotFoundError:
        print(f'{name:20s}  directory not found')

## Property scan

For each dataset, parse the first file to extract datagram types, channel IDs, sample counts, and data types. The `first_chrome` dataset contains files that lack standard EK80 datagram headers — likely an older or non-standard format — so it is skipped for detailed parsing.

In [ ]:
def scan_dataset(name, path, max_scan_pings=50):
    """Scan the first file in a dataset and return a summary dict."""
    files = list_raw_files(path)
    if not files:
        return {'name': name, 'n_files': 0, 'error': 'no .raw files'}

    f = files[0]
    info = {
        'name': name,
        'n_files': len(files),
        'first_file': f.name,
        'file_size_mb': f.stat().st_size / 1e6,
    }

    try:
        data = raw_parser.read_raw_file(str(f))
        types = raw_parser.find_datagram_types(data)
        info['datagram_types'] = {k: len(v) for k, v in types.items()}

        if 'RAW3' not in types:
            info['error'] = 'no RAW3 datagrams (non-standard format)'
            return info

        pings = ping_parser.extract_pings(str(f), max_pings=max_scan_pings)
        channels = defaultdict(list)
        for p in pings:
            channels[p.channel_id].append(p)

        info['channels'] = {}
        for ch_id, ch_pings in sorted(channels.items()):
            sample_counts = [p.sample_count for p in ch_pings]
            info['channels'][ch_id] = {
                'n_pings_sampled': len(ch_pings),
                'sample_count': int(np.median(sample_counts)),
                'sample_count_range': (min(sample_counts), max(sample_counts)),
                'data_type_raw': ch_pings[0].data_type_info['raw'],
                'has_complex': ch_pings[0].data_type_info['has_complex'],
                'has_angle': ch_pings[0].data_type_info['has_angle'],
            }
    except Exception as e:
        info['error'] = str(e)

    return info

# Scan existing reference
ref_info = scan_dataset('existing_training', EXISTING_REF)
print('=== Existing training data (reference) ===')
print(f"  Datagrams: {ref_info.get('datagram_types', {})}")
for ch, props in ref_info.get('channels', {}).items():
    print(f"  {ch}")
    print(f"    samples={props['sample_count']}, has_complex={props['has_complex']}, has_angle={props['has_angle']}")
print()

# Scan all new datasets
scan_results = {}
for name, path in DATASETS.items():
    info = scan_dataset(name, path)
    scan_results[name] = info
    print(f"=== {name} ({info['n_files']} files, {info.get('file_size_mb', 0):.0f} MB first file) ===")
    if 'error' in info:
        print(f"  ERROR: {info['error']}")
    if 'datagram_types' in info:
        print(f"  Datagrams: {info['datagram_types']}")
    for ch, props in info.get('channels', {}).items():
        print(f"  {ch}")
        print(f"    samples={props['sample_count']} ({props['sample_count_range'][0]}-{props['sample_count_range'][1]}), "
              f"has_complex={props['has_complex']}, has_angle={props['has_angle']}, raw_type={props['data_type_raw']}")
    print()

## Comparison summary

Key differences between datasets: number of channels/frequencies, sample depth, data encoding (complex vs power+angle), and presence of ADCP or noise-measurement channels.

In [ ]:
def extract_freq_label(channel_id):
    """Extract a short frequency label from a channel ID string."""
    parts = channel_id.split()
    for p in parts:
        for prefix in ['ES', 'EC', 'CP']:
            if p.startswith(prefix):
                base = p.split('_')[0]
                # For combo transducers like C-All, append the WBT sub-channel number
                if 'C-All' in channel_id or 'ADCP' in channel_id:
                    for q in parts:
                        if q.endswith(('-1', '-2', '-4', '-8', '-00', '-01', '-02', '-03', '-P0')):
                            base = f"{base}_ch{q.split('-')[-1]}"
                            break
                return base
    # Fallback: use the WBT serial + suffix
    return channel_id.replace(' ', '_')[:30]

print(f"{'Dataset':20s} {'Files':>5s} {'Channels':>8s} {'Frequencies':45s} {'Compat?':>8s}")
print('-' * 95)

ref_channels = set(ref_info.get('channels', {}).keys())

for name, info in [('EXISTING', ref_info)] + list(scan_results.items()):
    n_files = info.get('n_files', 0)
    channels = info.get('channels', {})
    if 'error' in info and not channels:
        freqs = info['error'][:45]
        compat = 'N/A'
    else:
        freq_labels = [extract_freq_label(ch) for ch in channels]
        freqs = ', '.join(sorted(set(freq_labels)))
        compat = 'YES' if set(channels.keys()) & ref_channels else 'no'
    print(f"{name:20s} {n_files:5d} {len(channels):8d} {freqs:45s} {compat:>8s}")

## Sample count and depth comparison

The sample count per ping determines the depth range of the echogram. Higher sample counts mean deeper range, which affects how patches should be extracted for training.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

all_entries = [('EXISTING', ref_info)] + [(n, scan_results[n]) for n in scan_results if 'channels' in scan_results[n]]
y_pos = 0
y_labels = []
y_positions = []

for ds_name, info in all_entries:
    channels = info.get('channels', {})
    for ch_id, props in sorted(channels.items()):
        freq = extract_freq_label(ch_id)
        label = f"{ds_name} / {freq}"
        ax.barh(y_pos, props['sample_count'], color='steelblue' if ds_name == 'EXISTING' else 'coral', height=0.7)
        ax.text(props['sample_count'] + 50, y_pos, str(props['sample_count']), va='center', fontsize=8)
        y_labels.append(label)
        y_positions.append(y_pos)
        y_pos += 1

ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels, fontsize=8)
ax.set_xlabel('Sample count per ping')
ax.set_title('Sample depth across datasets and channels')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUT_DIR / 'sample_count_comparison.png', dpi=150)
plt.show()

## Render all echograms as PNG

For each dataset, each `.raw` file is rendered as a power echogram (dB scale) and saved as a PNG. The output is organised as:
```
extra_raw_echograms/
  <dataset_name>/
    <frequency_label>/
      <filename>.png
```

Datasets with no parseable RAW3 datagrams (like `first_chrome`) are skipped.

In [ ]:
def render_echogram(raw_path, channel_id, out_path, vmin=-100, vmax=0):
    """Parse a .raw file and save the echogram for one channel as PNG."""
    pings = ping_parser.extract_pings(str(raw_path), channel_filter=channel_id)
    if len(pings) < 2:
        return False

    power_db = ping_parser.pings_to_power_image(pings, channel_id)
    if power_db.size == 0:
        return False

    power_db = np.clip(power_db, vmin, vmax)

    fig, ax = plt.subplots(figsize=(max(12, power_db.shape[1] / 80), 6))
    im = ax.imshow(power_db, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax,
                   interpolation='nearest')
    ax.set_xlabel('Ping number')
    ax.set_ylabel('Sample index')
    ax.set_title(f'{raw_path.name} — {channel_id}', fontsize=9)
    plt.colorbar(im, ax=ax, label='Power (dB)')
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    return True

In [ ]:
for ds_name, ds_path in DATASETS.items():
    info = scan_results.get(ds_name, {})
    channels = info.get('channels', {})
    if not channels:
        print(f"Skipping {ds_name} — no parseable channels")
        continue

    files = list_raw_files(ds_path)
    print(f"\n{'='*60}")
    print(f"Rendering {ds_name}: {len(files)} files × {len(channels)} channels")
    print(f"{'='*60}")

    for fi, raw_file in enumerate(files):
        # Re-extract channels per file since they may vary
        try:
            pings_sample = ping_parser.extract_pings(str(raw_file), max_pings=5)
            file_channels = sorted(set(p.channel_id for p in pings_sample))
        except Exception:
            print(f"  [{fi+1}/{len(files)}] {raw_file.name}: PARSE ERROR, skipping")
            continue

        for ch_id in file_channels:
            freq_label = extract_freq_label(ch_id)
            out_path = OUT_DIR / ds_name / freq_label / f"{raw_file.stem}.png"

            if out_path.exists():
                continue

            try:
                ok = render_echogram(raw_file, ch_id, out_path)
                status = 'OK' if ok else 'empty'
            except Exception as e:
                status = f'ERROR: {e}'

        print(f"  [{fi+1}/{len(files)}] {raw_file.name}: {len(file_channels)} channels rendered")

print("\nDone. Output in:", OUT_DIR)

## Output summary

List the generated folder structure and count of PNGs per dataset/frequency.

In [ ]:
for ds_dir in sorted(OUT_DIR.iterdir()):
    if not ds_dir.is_dir():
        continue
    print(f"\n{ds_dir.name}/")
    for freq_dir in sorted(ds_dir.iterdir()):
        if not freq_dir.is_dir():
            continue
        pngs = list(freq_dir.glob('*.png'))
        print(f"  {freq_dir.name}/  ({len(pngs)} PNGs)")

## Sample echograms from each dataset

Display one echogram per dataset (first available file) inline for quick visual comparison.

In [ ]:
for ds_name, ds_path in DATASETS.items():
    info = scan_results.get(ds_name, {})
    channels = info.get('channels', {})
    if not channels:
        continue

    files = list_raw_files(ds_path)
    raw_file = files[0]

    pings = ping_parser.extract_pings(str(raw_file), max_pings=500)
    file_channels = sorted(set(p.channel_id for p in pings))

    n_ch = len(file_channels)
    fig, axes = plt.subplots(1, n_ch, figsize=(6 * n_ch, 5), squeeze=False)

    for j, ch_id in enumerate(file_channels):
        power_db = ping_parser.pings_to_power_image(pings, ch_id)
        if power_db.size == 0:
            continue
        power_db = np.clip(power_db, -100, 0)
        ax = axes[0, j]
        im = ax.imshow(power_db, aspect='auto', cmap='viridis', vmin=-100, vmax=0)
        ax.set_title(f'{extract_freq_label(ch_id)}', fontsize=9)
        ax.set_xlabel('Ping')
        ax.set_ylabel('Sample')

    fig.suptitle(f'{ds_name}: {raw_file.name}', fontsize=11)
    plt.tight_layout()
    plt.show()

## Stitched echograms — all files per dataset

Each dataset folder likely contains consecutive segments of a single survey. Here we concatenate all files (sorted by filename/timestamp) into one continuous echogram per channel to check whether they form a seamless sequence.

In [ ]:
def stitch_dataset(ds_name, ds_path, channel_filter=None, max_pings_per_file=None):
    """Parse all .raw files in a dataset and concatenate into one image per channel."""
    files = list_raw_files(ds_path)
    if not files:
        return {}

    channel_images = defaultdict(list)

    for raw_file in files:
        try:
            kwargs = {}
            if max_pings_per_file:
                kwargs['max_pings'] = max_pings_per_file
            if channel_filter:
                kwargs['channel_filter'] = channel_filter
            pings = ping_parser.extract_pings(str(raw_file), **kwargs)
        except Exception:
            continue

        file_channels = sorted(set(p.channel_id for p in pings))
        for ch_id in file_channels:
            power_db = ping_parser.pings_to_power_image(pings, ch_id)
            if power_db.size > 0:
                channel_images[ch_id].append(power_db)

    stitched = {}
    for ch_id, images in channel_images.items():
        # Pad to uniform depth (max sample count) for concatenation
        max_depth = max(img.shape[0] for img in images)
        padded = []
        for img in images:
            if img.shape[0] < max_depth:
                pad = np.full((max_depth - img.shape[0], img.shape[1]), np.nan)
                img = np.vstack([img, pad])
            padded.append(img)
        stitched[ch_id] = np.hstack(padded)

    return stitched


def save_stitched_echogram(stitched, ds_name, out_dir, vmin=-100, vmax=0):
    """Save a stitched echogram as a wide PNG."""
    for ch_id, image in stitched.items():
        freq_label = extract_freq_label(ch_id)
        out_path = out_dir / f"{ds_name}_stitched_{freq_label}.png"

        width_inches = max(16, image.shape[1] / 100)
        fig, ax = plt.subplots(figsize=(width_inches, 6))

        display = np.where(np.isnan(image), vmin, np.clip(image, vmin, vmax))
        im = ax.imshow(display, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax,
                       interpolation='nearest')
        ax.set_xlabel('Ping number (all files concatenated)')
        ax.set_ylabel('Sample index')
        ax.set_title(f'{ds_name} — {freq_label} — stitched ({image.shape[1]} pings total)', fontsize=10)
        plt.colorbar(im, ax=ax, label='Power (dB)')
        plt.tight_layout()
        plt.savefig(out_path, dpi=120, bbox_inches='tight')
        plt.close(fig)
        print(f"  Saved: {out_path.name} ({image.shape[1]} x {image.shape[0]})")

    return out_path.parent

In [ ]:
STITCH_DIR = OUT_DIR / 'stitched'
STITCH_DIR.mkdir(exist_ok=True)

for ds_name, ds_path in DATASETS.items():
    info = scan_results.get(ds_name, {})
    channels = info.get('channels', {})
    if not channels:
        print(f"\nSkipping {ds_name} — no parseable channels")
        continue

    print(f"\n{'='*60}")
    print(f"Stitching {ds_name} ({info['n_files']} files)...")
    print(f"{'='*60}")

    stitched = stitch_dataset(ds_name, ds_path)
    if stitched:
        save_stitched_echogram(stitched, ds_name, STITCH_DIR)
    else:
        print("  No data to stitch")

print("\nAll stitched echograms saved to:", STITCH_DIR)

## Existing basto training data

The training data from the Bastø Electric vessel is spread across multiple download folders. We render individual file echograms and also stitch each folder into a continuous image to see the full survey coverage.

In [ ]:
BASTO_RAW_BASE = BASE / 'Master' / 'Datasets' / 'raw_files'
BASTO_OUT = OUT_DIR / 'basto_training'
BASTO_OUT.mkdir(exist_ok=True)

# Find all download folders with .raw files
basto_folders = sorted([
    d for d in BASTO_RAW_BASE.iterdir()
    if d.is_dir() and any(f.name.endswith('.raw') for f in d.iterdir() if f.is_file())
])

print(f"Found {len(basto_folders)} basto training folders:")
for folder in basto_folders:
    files = list_raw_files(folder)
    total_mb = sum(f.stat().st_size for f in files) / 1e6
    print(f"  {folder.name}: {len(files)} files, {total_mb:.0f} MB")

In [ ]:
# Render individual file echograms for the basto training data
for folder in basto_folders:
    folder_label = folder.name
    files = list_raw_files(folder)
    print(f"\n{'='*60}")
    print(f"Rendering basto training: {folder_label} ({len(files)} files)")
    print(f"{'='*60}")

    for fi, raw_file in enumerate(files):
        try:
            pings_sample = ping_parser.extract_pings(str(raw_file), max_pings=5)
            file_channels = sorted(set(p.channel_id for p in pings_sample))
        except Exception:
            print(f"  [{fi+1}/{len(files)}] {raw_file.name}: PARSE ERROR")
            continue

        for ch_id in file_channels:
            freq_label = extract_freq_label(ch_id)
            out_path = BASTO_OUT / folder_label / freq_label / f"{raw_file.stem}.png"

            if out_path.exists():
                continue

            try:
                ok = render_echogram(raw_file, ch_id, out_path)
            except Exception:
                pass

        print(f"  [{fi+1}/{len(files)}] {raw_file.name}: {len(file_channels)} channels")

print("\nBasto individual echograms saved to:", BASTO_OUT)

In [ ]:
# Stitch each basto training folder into one continuous echogram
BASTO_STITCH_DIR = OUT_DIR / 'basto_training_stitched'
BASTO_STITCH_DIR.mkdir(exist_ok=True)

for folder in basto_folders:
    folder_label = folder.name
    print(f"\nStitching {folder_label}...")
    stitched = stitch_dataset(folder_label, folder)
    if stitched:
        save_stitched_echogram(stitched, folder_label, BASTO_STITCH_DIR)

# Also stitch ALL basto folders together as one mega-echogram (sorted by filename = timestamp order)
print(f"\n{'='*60}")
print("Stitching ALL basto training files into one continuous echogram...")
print(f"{'='*60}")

all_basto_files = sorted([
    f for folder in basto_folders for f in list_raw_files(folder)
])
print(f"Total: {len(all_basto_files)} files")

# Create a temporary directory-like object that list_raw_files won't need
# Instead, directly build the stitched image
channel_images = defaultdict(list)
for raw_file in all_basto_files:
    try:
        pings = ping_parser.extract_pings(str(raw_file))
    except Exception:
        continue
    file_channels = sorted(set(p.channel_id for p in pings))
    for ch_id in file_channels:
        power_db = ping_parser.pings_to_power_image(pings, ch_id)
        if power_db.size > 0:
            channel_images[ch_id].append(power_db)

basto_full_stitched = {}
for ch_id, images in channel_images.items():
    max_depth = max(img.shape[0] for img in images)
    padded = []
    for img in images:
        if img.shape[0] < max_depth:
            pad = np.full((max_depth - img.shape[0], img.shape[1]), np.nan)
            img = np.vstack([img, pad])
        padded.append(img)
    basto_full_stitched[ch_id] = np.hstack(padded)

save_stitched_echogram(basto_full_stitched, 'basto_ALL_training', BASTO_STITCH_DIR)
print("\nBasto stitched echograms saved to:", BASTO_STITCH_DIR)

In [ ]:
# Display the stitched results inline (downsampled for notebook rendering)
from PIL import Image

stitch_pngs = sorted(STITCH_DIR.glob('*.png')) + sorted(BASTO_STITCH_DIR.glob('*.png'))
print(f"Total stitched PNGs: {len(stitch_pngs)}\n")

for png_path in stitch_pngs:
    img = Image.open(png_path)
    w, h = img.size
    # Downsample wide images for display
    if w > 2400:
        scale = 2400 / w
        img = img.resize((2400, int(h * scale)), Image.LANCZOS)

    fig, ax = plt.subplots(figsize=(16, 4))
    ax.imshow(np.array(img))
    ax.set_axis_off()
    ax.set_title(png_path.name, fontsize=9)
    plt.tight_layout()
    plt.show()